# Power Curve Visualization

Loads simulation results from S3 and plots the empirical power curve across sample sizes.

Run this notebook in a SageMaker JupyterLab instance with access to the `trial-sim-results` S3 bucket.

**Steps:**
1. Install packages
2. List available runs for today
3. Select run and pull data from S3
4. Plot power curve
5. Print summary

## Install packages

In [ ]:
!pip install pandas matplotlib boto3 -q

## List available runs for today

In [ ]:
import boto3
from datetime import datetime

s3 = boto3.client("s3")
bucket = "trial-sim-results"
date_prefix = datetime.now().strftime("%Y%m%d")

# List all job ID subfolders under today's date
response = s3.list_objects_v2(Bucket=bucket, Prefix=f"results/{date_prefix}/", Delimiter="/")

job_folders = [cp["Prefix"] for cp in response.get("CommonPrefixes", [])]

print(f"Available runs for {date_prefix}:")
for i, folder in enumerate(job_folders):
    print(f"  [{i}] {folder}")

## Select run and pull data from S3

In [ ]:
import pandas as pd
import io

# <- SET THIS to the index printed above
selected_index = 0

selected_folder = job_folders[selected_index]
print(f"Loading results from: {selected_folder}")

# Pull all CSVs from selected job folder
response = s3.list_objects_v2(Bucket=bucket, Prefix=selected_folder)
files = [obj["Key"] for obj in response["Contents"] if obj["Key"].endswith(".csv")]

print(f"Found {len(files)} result files")

frames = []
for key in files:
    obj = s3.get_object(Bucket=bucket, Key=key)
    df = pd.read_csv(io.BytesIO(obj["Body"].read()))
    frames.append(df)

results = pd.concat(frames).sort_values("n_per_arm").reset_index(drop=True)
print(f"\nTotal scenarios loaded: {len(results)}")
print(results.to_string(index=False))

## Plot power curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(results["n_per_arm"], results["power"],
         marker="o", color="steelblue", linewidth=2, label="Empirical Power")
plt.axhline(y=0.80, color="red", linestyle="--", linewidth=1.5, label="80% Power Threshold")

min_n_row = results[results["power"] >= 0.80].iloc[0]
plt.axvline(x=min_n_row["n_per_arm"], color="green", linestyle="--",
            linewidth=1.5, label=f"Minimum n = {int(min_n_row['n_per_arm'])}")

plt.xlabel("N per arm (randomized)", fontsize=13)
plt.ylabel("Empirical Power", fontsize=13)
plt.title("Power Simulation: Drug A vs Placebo\n"
          "Effect size=4kg, SD=4.5kg, 20% dropout, 10,000 iterations per scenario",
          fontsize=13)
plt.ylim(0, 1.05)
plt.xticks(results["n_per_arm"], rotation=45)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("power_curve.png", dpi=150)
plt.show()

## Print summary

In [ ]:
min_n_row = results[results["power"] >= 0.80].iloc[0]

print(f"Minimum n per arm achieving >= 80% power: {int(min_n_row['n_per_arm'])}")
print(f"Empirical power at that n:                {min_n_row['power']:.4f}")
print(f"Enrolled per arm after 20% dropout inflation: {int(min_n_row['n_per_arm'] / 0.80)}")
print(f"Total enrollment both arms:               {int(min_n_row['n_per_arm'] / 0.80) * 2}")